# **Setup**

In [23]:
import pandas as pd 
import regex as re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet',quiet=True)

from sklearn.feature_extraction.text import TfidfVectorizer

# **Data Import**

In [2]:
data = {'text': ["I love cooking!", "Baking is fun", None, "Japanese cuisine is great!"]}

df = pd.DataFrame(data)
print(df)

                         text
0             I love cooking!
1               Baking is fun
2                        None
3  Japanese cuisine is great!


# **Data Cleaning**

In [3]:
df.dropna(subset=['text'], inplace=True)
print(df)

                         text
0             I love cooking!
1               Baking is fun
3  Japanese cuisine is great!


# **Data Normalization**

In [4]:
df['text'] = df['text'].str.lower()
print(df)

                         text
0             i love cooking!
1               baking is fun
3  japanese cuisine is great!


# **Remove Noise**

In [5]:
df['text'] = df['text'].apply(lambda x: re.sub(r'[^\w\s]', '', x))
print(df)

                        text
0             i love cooking
1              baking is fun
3  japanese cuisine is great


# **Tokenization**

In [6]:
df['tokens'] = df['text'].str.split()
print(df)

                        text                          tokens
0             i love cooking              [i, love, cooking]
1              baking is fun               [baking, is, fun]
3  japanese cuisine is great  [japanese, cuisine, is, great]


# **Remove Stopwords**

In [14]:
# nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))
print(stop_words)

{'his', 'y', 'until', 'a', 'him', 'ma', 'there', 'weren', "you'd", 'down', 'hers', 'themselves', 'doesn', "you'll", 'ourselves', 'during', 'should', 'myself', 'her', 'you', 'after', 'over', 'off', 'can', 'what', 'before', 'don', 'll', 'o', "you've", 'did', 'for', 'these', 'herself', "aren't", 'they', 'whom', "haven't", "isn't", 'she', 'and', 'but', 'in', "hasn't", 'so', 'very', "hadn't", 'mightn', 'against', 'to', 'our', 'about', 'will', "should've", 'most', 'of', 'hadn', 'd', "she's", 'because', 'them', 'am', 'shan', 'than', 'by', 'no', "mightn't", 'each', 'nor', 'm', 'an', 'himself', 'he', 'do', 'only', 'been', 'if', 'below', 'shouldn', 'on', 'above', "doesn't", "didn't", 'again', 'be', 's', "wasn't", 'through', 'does', 'few', 'yourself', 'such', 'more', 'own', 'who', 'now', 'into', 'while', 'when', 'yourselves', 'has', 'being', 'how', 'why', 'same', 'this', "shouldn't", 'my', 'then', 'between', 'it', 'its', 'not', "couldn't", 'is', "don't", 'which', 'just', 'isn', 'i', 'the', 'didn'

In [15]:
# Remove Stopwords
df['tokens'] = df['tokens'].apply(lambda x: [word for word in x if word not in stop_words])

print(df['tokens'])

0               [love, cooking]
1                 [baking, fun]
3    [japanese, cuisine, great]
Name: tokens, dtype: object


# **Stemming & Lemmatization**

In [21]:
# Create a stemmer object
stemmer = PorterStemmer()

df['stemmed'] = df['tokens'].apply(lambda x: [stemmer.stem(word) for word in x])
print(df[['tokens', 'stemmed']])

                       tokens                   stemmed
0             [love, cooking]              [love, cook]
1               [baking, fun]               [bake, fun]
3  [japanese, cuisine, great]  [japanes, cuisin, great]


# **Vectorization**

In [24]:
df

,text,tokens,stemmed
0,i love cooking,"[love, cooking]","[love, cook]"
1,baking is fun,"[baking, fun]","[bake, fun]"
3,japanese cuisine is great,"[japanese, cuisine, great]","[japanes, cuisin, great]"


## **TFIDF Vectorizer**

In [25]:
df['text_cleaned'] = df['tokens'].apply(lambda x: ' '.join(x))
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df['text_cleaned'])
print(X.toarray())

[[0.         0.70710678 0.         0.         0.         0.
  0.70710678]
 [0.70710678 0.         0.         0.70710678 0.         0.
  0.        ]
 [0.         0.         0.57735027 0.         0.57735027 0.57735027
  0.        ]]


In [26]:
X.shape

(3, 7)

## **Word2Vec Vectorizer**

In [31]:
from gensim.models import Word2Vec

# Train a Word2Vec model
model = Word2Vec(df['tokens'], vector_size=100, window=5, min_count=1, workers=4)
model

In [34]:
# Convert a word to a vector
word = "cooking"

# Get the vector for the word
vector = model.wv[word]

print()
print(vector)

[-8.7274825e-03  2.1301615e-03 -8.7354420e-04 -9.3190884e-03
 -9.4281426e-03 -1.4107180e-03  4.4324086e-03  3.7040710e-03
 -6.4986930e-03 -6.8730675e-03 -4.9994122e-03 -2.2868442e-03
 -7.2502876e-03 -9.6033178e-03 -2.7436293e-03 -8.3628409e-03
 -6.0388758e-03 -5.6709289e-03 -2.3441375e-03 -1.7069972e-03
 -8.9569986e-03 -7.3519943e-04  8.1525063e-03  7.6904297e-03
 -7.2061159e-03 -3.6668312e-03  3.1185520e-03 -9.5707225e-03
  1.4764392e-03  6.5244664e-03  5.7464195e-03 -8.7630618e-03
 -4.5171441e-03 -8.1401607e-03  4.5956374e-05  9.2636338e-03
  5.9733056e-03  5.0673080e-03  5.0610625e-03 -3.2429171e-03
  9.5521836e-03 -7.3564244e-03 -7.2703874e-03 -2.2653891e-03
 -7.7856064e-04 -3.2161034e-03 -5.9258583e-04  7.4888230e-03
 -6.9751858e-04 -1.6249407e-03  2.7443992e-03 -8.3591007e-03
  7.8558037e-03  8.5361041e-03 -9.5840869e-03  2.4462664e-03
  9.9049713e-03 -7.6658037e-03 -6.9669187e-03 -7.7365171e-03
  8.3959233e-03 -6.8133592e-04  9.1444086e-03 -8.1582209e-03
  3.7430846e-03  2.63504

## **BERT**

In [1]:
from transformers import BertTokenizer, BertModel

# Load model
bert_vectorizer = BertModel.from_pretrained('bert-base-uncased')

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

In [ ]:
bert_vectorizer.

# **References**

- https://www.kdnuggets.com/cleaning-and-preprocessing-text-data-in-pandas-for-nlp-tasks 